In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
    
GOOGLE_MODEL = "gemini-3.1-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL, 
    temperature=0,
)

In [10]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser 
from langchain_core.prompts import ChatPromptTemplate

class Sentiment(Enum):
    """리뷰 감성 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"

parser = EnumOutputParser(enum=Sentiment)

format_instructions = parser.get_format_instructions()

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 리뷰 감정 분석 분류 전문가입니다. 리뷰를 읽고 감성을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | parser

In [12]:
from langchain_core.exceptions import OutputParserException

reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
]

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = chain.invoke({"review": review})

        print(f"  분류 결과: {result} / 타입: {type(result).__name__} / 값: {result.value}")
    except OutputParserException as e:
        print(f"  파싱 오류: {e}")
    print("-" * 50)

리뷰: 이 영화 정말 재미없어요. 시간 낭비였습니다.
  분류 결과: Sentiment.NEGATIVE / 타입: Sentiment / 값: 부정
--------------------------------------------------
리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
  분류 결과: Sentiment.POSITIVE / 타입: Sentiment / 값: 긍정
--------------------------------------------------
리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
  분류 결과: Sentiment.NEUTRAL / 타입: Sentiment / 값: 보통
--------------------------------------------------
